In [16]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# t_res='5min'; res='1km'
# res='1km'
# Np_str='1e6'

# # dx = 1 km; Np = 1M; Nt = 1 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_5min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_5min.nc') #***
# t_res='5min'; res='1km'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_50M_5min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_50M_5min.nc') #***
res='1km'; t_res='1min'
Np_str='50e6'


# # dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***
# res='250m'
# Np_str='150e6'

In [27]:
#INITIALIZE DATA FUNCTION
###############################################################
def initiate_array(out_file, vars, t_chunk_size, p_chunk_size, t_size=None, p_size=None):
    if t_size is None:
        t_size = len(data['time'])  # Number of timesteps
    if p_size is None:
        p_size = len(parcel['xh'])  # Number of vertical levels

    with h5py.File(out_file, 'w') as f:
        for var_name in vars:
            if var_name not in f:
                # Set dtype conditionally
                if var_name in ['Z', 'Y', 'X']:
                    dtype = np.uint16
                elif var_name in ['A_g', 'A_c']:
                    dtype = np.bool_
                else:
                    dtype = np.float32  # or whatever your default is

                f.create_dataset(
                    var_name,
                    shape=(t_size, p_size),
                    chunks=(t_chunk_size, p_chunk_size),
                    dtype=dtype
                )

In [9]:
# block = 10  # number of time steps to write at a time
# for start in range(0, t_size, block):
#     end = min(start + block, t_size)
#     size = (end - start, p_size)

In [17]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [41]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=180 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(data['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=7
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 12, end_job = 14


In [42]:
#Indexing Array with JobArray
data=data.isel(time=slice(start_job,end_job))
parcel=parcel.isel(time=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [10]:
###########################################################################################################################################################################

In [11]:
#MAKING LAGRANGIAN BINARY ARRAY
###############################################################

In [35]:
#Lagrangian Position Arrays
##############
def grid_location(z,y,x):
    zf=data['zf'].values*1000; which_zh=np.clip(np.searchsorted(zf,z)-1,0,None).astype(np.uint16)
    #which_zh=np.where(which_zh == -1, 0, which_zh) 
    
    yf=data['yf'].values*1000; which_yh=np.clip(np.searchsorted(yf,y)-1,0,None).astype(np.uint16) 
    #which_yh=np.where(which_yh == -1, 0, which_yh) 
    
    xf=data['xf'].values*1000; which_xh=np.clip(np.searchsorted(xf,x)-1,0,None).astype(np.uint16)
    #which_xh=np.where(which_xh == -1, 0, which_xh) 
    
    return which_zh,which_yh,which_xh

print('Loading Lagrangian x,y,z into Memory\n')
x=parcel['x'].data;y=parcel['y'].data;z=parcel['z'].data

print('Creating Lagrangian X,Y,Z Binary Arrays\n')
Z,Y,X=grid_location(z,y,x)

check_memory()

Loading Lagrangian x,y,z into Memory

Creating Lagrangian X,Y,Z Binary Arrays

Top 10 objects with highest memory usage
{'A_g': '810.45 MB', 'A_c': '810.45 MB', 'Z': '202.61 MB', 'Y': '202.61 MB', 'X': '202.61 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'out_file': '0.0 MB'}

2.22873 GB in use overall


In [16]:
def call_variable(varname):
    var_data=data[varname].data
    return var_data

In [17]:
def make_lagrangian_array(varname):
    var_data=call_variable(varname)
    VAR=np.zeros_like(Z,dtype='float32')
    
    Nt=len(data['time'])
    Np=len(parcel['xh'])
    for p in np.arange(Np):
        if np.mod(p,1e6)==0: print(f"{p}/{len(parcel['xh'])}")
    
        #Get Indicies
        zs=Z[:,p]
        ys=Y[:,p]
        xs=X[:,p]
        ts = np.arange(Nt)  
    
        #Get Values
        vars = var_data[ts, zs, ys, xs]

        #Adding to Lagrangian Array
        VAR[:,p]=vars
        del vars
        
    del var_data
    return VAR

In [18]:
print('Making W, QC, and QI Lagrangian Array\n')

W=make_lagrangian_array('winterp'); check_memory() 
QC=make_lagrangian_array('qc'); check_memory()
QI = make_lagrangian_array('qi'); check_memory()

print('Making QC+QI Lagrangian Array\n')
import dask.array as da
Nt=len(data['time'])
QC = da.from_array(QC, chunks=(Nt, 'auto'))
QI = da.from_array(QI, chunks=(Nt, 'auto'))
QCQI=QC+QI
QCQI=QCQI.compute(); check_memory()
array_to_dask=True

Making W, QC, and QI Lagrangian Array

0/50653000
1000000/50653000
2000000/50653000
3000000/50653000
4000000/50653000
5000000/50653000
6000000/50653000
7000000/50653000
8000000/50653000
9000000/50653000
10000000/50653000
11000000/50653000
12000000/50653000
13000000/50653000
14000000/50653000
15000000/50653000
16000000/50653000
17000000/50653000
18000000/50653000
19000000/50653000
20000000/50653000
21000000/50653000
22000000/50653000
23000000/50653000
24000000/50653000
25000000/50653000
26000000/50653000
27000000/50653000
28000000/50653000
29000000/50653000
30000000/50653000
31000000/50653000
32000000/50653000
33000000/50653000
34000000/50653000
35000000/50653000
36000000/50653000
37000000/50653000
38000000/50653000
39000000/50653000
40000000/50653000
41000000/50653000
42000000/50653000
43000000/50653000
44000000/50653000
45000000/50653000
46000000/50653000
47000000/50653000
48000000/50653000
49000000/50653000
50000000/50653000
Top 10 objects with highest memory usage
{'Z': '810.45 MB',

In [21]:
#Create Set Thresholds and Create Binary Arrays
w_thresh1=0.1
w_thresh2=0.5

qcqi_thresh1=1e-6
qcqi_thresh2=1e-6

print('Making Lagrangian Binary Arrays\n')
#Apply Thresholds 
if array_to_dask==True:
    import dask.array as da
    Nt=len(data['time'])
    W = da.from_array(W, chunks=(Nt, 'auto'))
    QCQI = da.from_array(QCQI, chunks=(Nt, 'auto'))
    array_to_dask=False

print('Making A_g')
A_g = np.where((W >= w_thresh1) & (QCQI < qcqi_thresh1), True, False)
print('Making A_c')
A_c = np.where((W >= w_thresh2) & (QCQI >= qcqi_thresh2), True, False)
A_g=A_g.compute(); A_c=A_c.compute()
check_memory()

Making Lagrangian Binary Arrays

Making A_g
Making A_c
Top 10 objects with highest memory usage
{'z': '405.22 MB', 'Z': '202.61 MB', 'Y': '202.61 MB', 'X': '202.61 MB', 'A_g': '101.31 MB', 'A_c': '101.31 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'in_file': '0.0 MB'}

1.21567 GB in use overall


In [ ]:
# Saving Data
##############
print('Saving Data\n')
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
# out_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min_{job_id}.h5' 
out_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_1min_50M_{job_id}.h5' 

vars=['z','x','Z','Y','X','W','QCQI','A_g','A_c']
initiate_array(out_file,vars,t_chunk_size=1,p_chunk_size=100_000)
with h5py.File(out_file, 'a') as f: 
    f['z'][:]=z
    f['x'][:]=x
    
    f['Z'][:]=Z
    f['Y'][:]=Y
    f['X'][:]=X

    f['W'][:]=W
    f['QCQI'][:]=QCQI
    
    f['A_g'][:]=A_g
    f['A_c'][:]=A_c

In [5]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOB ARRAY IS RUNNING
recombine=True

In [ ]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
    dir3=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
    out_file=dir3+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5' 5' 

    print('initializing array')
    vars=['z','x','Z','Y','X','W','QCQI','A_g','A_c']
    initiate_array(out_file,vars,t_chunk_size=100,p_chunk_size=100_000)

    print('recombining')
    with h5py.File(out_file, 'r+') as f_out:
        num_jobs=360
        # for job_id in np.arange(1,num_jobs+1): #OG
        for job_id in np.arange(1,2+1): #TESTING
            if np.mod(job_id,5)==0: print(f"job_id = {job_id}")
            [a,b] = get_job_range(job_id,num_jobs)
    
            # in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min_{job_id}.h5' 
            in_file = dir2 + f'lagrangian_binary_array_{res}_{t_res}_{Np_str}_{job_id}.h5' 
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]
            break #TESTING

In [ ]:
#DASK-ENABLED
def recombine(in_files,out_file):
    matching_files = sorted(glob.glob(in_files))
    print('recombining')
    # %%time
    out=xr.open_mfdataset(matching_files,engine='h5netcdf',concat_dim='phony_dim_0',combine='nested',phony_dims='sort')
    out.to_netcdf(out_file, engine='h5netcdf')
    
if recombine==True:
    import glob
    dir3=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
    in_files = dir2 + f'lagrangian_binary_array_{res}_{t_res}_{Np_str}_*.h5'
    out_file=dir3+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5' 
    
    recombine(in_files,out_file)

In [6]:
#########################################
# Reading Back Data Later

In [9]:
# ##############
# import h5py
# # dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/job_out/'
# in_file=dir3+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
# with h5py.File(in_file, 'r') as f:
#     # Load the dataset by its name
#     A_g = f['A_g'][:]
#     A_c = f['A_c'][:]

#     W = f['W'][:]
#     QCQI = f['QCQI'][:]
#     z = f['z'][:]
#     Z = f['Z'][:]
#     Y = f['Y'][:]
#     X = f['X'][:]

# # # #Making Time Matrix
# # # rows, cols = A.shape[0], A.shape[1]
# # # T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [21]:
# #TESTING CHECKING THAT THRESHOLDS WORKED
# # w_data=data['winterp'].data #interpolation w data z coordinate from zh to zf
# # variable='qc'; qc_data=data[variable].data # get qc data
# # variable='qi'; qi_data=data[variable].data # get qc data
# # qc_plus_qi=qc_data+qi_data

# def test_thresholds(t,type):
#     # w_data=data['w'].interp(zf=data['zh']).data

#     if type=='g':
#         A=A_g
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'max qcqi is {hey.max()}')
#     if type=='c':
#         A=A_c
#         where=np.where(A[t]==1)
#         hey=w_data[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min w is {hey.min()}')
        
#         # qc_plus_qi=(data['qc']+data['qi']).data
#         hey=qc_plus_qi[t,Z[t,where],Y[t,where],X[t,where]]
#         print(f'min qcqi is {hey.min()}')

# #GENERAL UPDRAFT
# t=70
# test_thresholds(t,'g')
# print('\n')
# #CLOUDY UPDRAFT
# test_thresholds(t,'c')

min w is 0.10000452399253845
max qcqi is 9.999595595999722e-10


min w is 0.5002251863479614
min qcqi is 1.096519213206193e-06
